# 07 End-to-End Maintenance Pipeline & Export

This notebook combines raw data handling, data cleaning, ordinal encoding, physics-based feature engineering, scaling, and classification into a single Scikit-Learn `Pipeline` using `FunctionTransformer`, and exports the trained pipeline to a single `.pkl` file.

## 1. Import Libraries & Load Dataset

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from lightgbm import LGBMClassifier

sys.path.append(os.path.abspath(".."))
from src.data_loader import load_ai4i_data

df = load_ai4i_data()
X = df.drop(columns=['machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf'])
y = df['machine_failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

Training set shape: (8000, 8), Testing set shape: (2000, 8)


## 2. Define Transformer Functions

In [2]:
def preprocess_cleaning(df_in):
    X_c = df_in.copy()
    drop_cols = [c for c in ['udi', 'product_id'] if c in X_c.columns]
    if drop_cols:
        X_c = X_c.drop(columns=drop_cols)
    type_map = {'L': 0, 'M': 1, 'H': 2}
    if 'type' in X_c.columns:
        X_c['type'] = X_c['type'].map(type_map).fillna(X_c['type'])
    return X_c

def feature_engineering(df_in):
    X_fe = df_in.copy()
    if 'rotational_speed_rpm' in X_fe.columns:
        X_fe['log_rotational_speed'] = np.log1p(X_fe['rotational_speed_rpm'])
        rpm = X_fe['rotational_speed_rpm']
    else:
        rpm = np.expm1(X_fe['log_rotational_speed'])
        
    X_fe['temp_diff'] = X_fe['process_temperature_k'] - X_fe['air_temperature_k']
    X_fe['power_w'] = X_fe['torque_nm'] * (rpm * 2 * np.pi / 60)
    X_fe['tool_wear_torque'] = X_fe['tool_wear_min'] * X_fe['torque_nm']
    
    if 'rotational_speed_rpm' in X_fe.columns:
        X_fe = X_fe.drop(columns=['rotational_speed_rpm'])
        
    return X_fe

## 3. Build & Train End-to-End Pipeline

In [3]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1
)

pipeline = Pipeline(steps=[
    ('cleaning', FunctionTransformer(preprocess_cleaning)),
    ('feature_engineering', FunctionTransformer(feature_engineering)),
    ('scaler', StandardScaler()),
    ('classifier', model)
])

pipeline.fit(X_train, y_train)
print("Pipeline training completed successfully.")

Pipeline training completed successfully.


## 4. Evaluate Pipeline Performance

In [4]:
acc = pipeline.score(X_test, y_test)
print(f"Test Accuracy: {acc:.4f}")

Test Accuracy: 0.9720


## 5. Export Pipeline to PKL File

In [5]:
model_dir = '../models'
if not os.path.exists(model_dir):
    model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)
export_path = os.path.join(model_dir, 'smart_maintenance_pipeline.pkl')
joblib.dump(pipeline, export_path)
print(f"Pipeline exported successfully to: {export_path}")

Pipeline exported successfully to: models\smart_maintenance_pipeline.pkl


## 6. Raw Data Inference Verification

In [6]:
sample_raw_data = pd.DataFrame([{
    'udi': 1,
    'product_id': 'M14860',
    'type': 'M',
    'air_temperature_k': 298.1,
    'process_temperature_k': 308.6,
    'rotational_speed_rpm': 1551,
    'torque_nm': 42.8,
    'tool_wear_min': 0
}])

loaded_pipeline = joblib.load(export_path)
pred_class = loaded_pipeline.predict(sample_raw_data)[0]
pred_proba = loaded_pipeline.predict_proba(sample_raw_data)[0]

print(f"Predicted Class (0: Normal, 1: Failure): {pred_class}")
print(f"Prediction Probabilities [Normal, Failure]: {pred_proba}")

Predicted Class (0: Normal, 1: Failure): 0
Prediction Probabilities [Normal, Failure]: [0.99779215 0.00220785]
